In [1]:
import os
import random
import shutil
from collections import defaultdict, Counter
from pathlib import Path

from tqdm import tqdm
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [2]:
# Sliding-window configuration used for crop analysis
WINDOW = 512
STRIDE = 512
IGNORE_INDEX = -1
TAMPER_THR = 0.0

In [ ]:
# Base path for the RTM dataset
REALTEXT_ROOT = "/media/general_storage6/rmastorage/datasets/RealTextManipulation"

# Raw image and mask directories
IMG_DIR = os.path.join(REALTEXT_ROOT, "JPEGImages")
MASK_DIR = os.path.join(REALTEXT_ROOT, "SegmentationClass")

REALTEXT_ROOT = Path(REALTEXT_ROOT)
IMG_DIR = Path(IMG_DIR)
MASK_DIR = Path(MASK_DIR)

In [4]:
random.seed(42)

# Document type prefixes used in RTM
TYPES = ["cover_", "cpmv_", "good_", "edit_", "inpaint_", "insert_", "splice_"]

In [ ]:
def list_pairs(jpeg_dir, gt_dir):
    """
    List all valid (image, mask) pairs in the raw dataset.
    An image is considered valid if a mask with the same stem exists.
    """
    
    print("Reading all JPEG + mask pairs...")
    exts = ["*.jpg", "*.png", "*.jpeg", "*.JPG", "*.PNG", "*.JPEG"]
    pairs = []

    for ext in exts:
        for img in jpeg_dir.glob(ext):
            mask = gt_dir / (img.stem + ".png")
            if mask.exists():
                pairs.append((img, mask))

    print(f"  Total valid pairs found: {len(pairs)}")
    return pairs

In [6]:
pairs = list_pairs(IMG_DIR, MASK_DIR)

🔍 A ler todos os pares JPEG + máscara...
   ➤ Total de pares encontrados: 9000


In [ ]:
def load_document_by_name(
    image_dir: Path,
    mask_dir: Path,
    train_image_name: str,
    image_exts=(".jpg", ".jpeg", ".png"),
    mask_ext=".png",
):
    """
    Load one image and its corresponding mask by stem name.
    The mask is binarized into {0,1}.
    """
    
    img_path = None
    for ext in image_exts:
        p = image_dir / f"{train_image_name}{ext}"
        if p.exists():
            img_path = p
            break

    if img_path is None:
        raise FileNotFoundError(f"Image '{train_image_name}' not found in {image_dir}")

    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    mask_path = mask_dir / f"{train_image_name}{mask_ext}"
    if mask_path.exists():
        mask_np = np.array(Image.open(mask_path)).astype(np.int16)
        mask_np = (mask_np > 0).astype(np.int16)
    else:
        # Safe fallback: assume clean document if mask is missing
        mask_np = np.zeros(img_np.shape[:2], dtype=np.int16)

    return img_np, mask_np, img_path

In [ ]:
def overlay_mask_on_rgb(
    img_rgb: np.ndarray,
    mask: np.ndarray,
    alpha: float = 0.85,
    color_rgb=(255, 0, 0),
    ignore_value: int = -1,
    dilate_px: int = 7,
    show_contours: bool = True,
    contour_thickness: int = 2,
):
    """
    Overlay a binary tampering mask on top of an RGB image for visualization.
    """
    
    assert img_rgb.ndim == 3 and img_rgb.shape[2] == 3
    H, W = img_rgb.shape[:2]
    if mask.shape[:2] != (H, W):
        raise ValueError(f"mask shape {mask.shape} != image {(H, W)}")

    m = mask.copy()

    if m.dtype == np.bool_:
        pos = m
    else:
        valid = (m != ignore_value) if np.any(m == ignore_value) else np.ones_like(m, dtype=bool)
        pos = (m > 0) & valid

    pos_u8 = (pos.astype(np.uint8) * 255)

    if dilate_px and dilate_px > 0:
        k = 2 * dilate_px + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        pos_u8 = cv2.dilate(pos_u8, kernel, iterations=1)

    pos2 = pos_u8.astype(bool)

    out = img_rgb.copy().astype(np.float32)
    color = np.array(color_rgb, dtype=np.float32).reshape(1, 1, 3)
    out[pos2] = (1 - alpha) * out[pos2] + alpha * color
    out = np.clip(out, 0, 255).astype(np.uint8)

    if show_contours:
        contours, _ = cv2.findContours(pos_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, (255, 255, 0), contour_thickness)

    return out

In [9]:
def show_full_image_mask_overlay(
    img_rgb: np.ndarray,
    mask: np.ndarray,
    title="",
    alpha=0.85,
    dilate_px=7,
    figsize=(18, 6),
):
    """
    Display image, mask, and overlay side by side.
    """
    
    overlay = overlay_mask_on_rgb(img_rgb, mask, alpha=alpha, dilate_px=dilate_px)

    fig, ax = plt.subplots(1, 3, figsize=figsize)
    ax[0].imshow(img_rgb)
    ax[0].set_title(f"Image {title}")
    ax[0].axis("off")

    ax[1].imshow(mask, cmap="gray")
    ax[1].set_title(f"Mask {title}")
    ax[1].axis("off")

    ax[2].imshow(overlay)
    ax[2].set_title(f"Overlay (alpha={alpha}, dilate={dilate_px}) {title}")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()

In [10]:
def infer_type_from_stem(stem: str, types=TYPES):
    """
    Infer document type from filename prefix.
    """
    
    for t in types:
        if stem.startswith(t):
            return t
    return "other"

In [ ]:
def sliding_origins(H, W, win=512, stride=512):
    """
    Generate sliding-window origins that fully cover the image.
    Ensures the final window is aligned to the image border when needed.
    """
    
    if H <= win:
        ys = [0]
    else:
        ys = list(range(0, H - win + 1, stride))
        if ys[-1] != H - win:
            ys.append(H - win)

    if W <= win:
        xs = [0]
    else:
        xs = list(range(0, W - win + 1, stride))
        if xs[-1] != W - win:
            xs.append(W - win)

    return [(y, x) for y in ys for x in xs]

In [ ]:
# Example document for visualization
train_image_name = "cpmv_0356"

train_image_full, train_mask_full, train_image_path = load_document_by_name(
    image_dir=IMG_DIR,
    mask_dir=MASK_DIR,
    train_image_name=train_image_name,
)

In [ ]:
show_full_image_mask_overlay(
    train_image_full,
    train_mask_full,
    title=train_image_name,
    alpha=0.90,
    dilate_px=0,
)

In [ ]:
H, W = train_image_full.shape[:2]
origins = sliding_origins(H, W, win=WINDOW, stride=STRIDE)

print(f"Image size: H={H}, W={W}")
print(f"Window={WINDOW}, stride={STRIDE}")
print(f"Num crops: {len(origins)}")
print("First 10 origins:", origins[:10])
print("Last 10 origins:", origins[-10:])

In [ ]:
# Binary mask for visualization
mask_vis = (train_mask_full > 0).astype(np.uint8)

# RGB image with crop windows
fig_rgb, ax_rgb = plt.subplots(figsize=(10, 10))
ax_rgb.imshow(train_image_full)
ax_rgb.axis("off")

for i, (y, x) in enumerate(origins):
    rect = patches.Rectangle(
        (x, y), WINDOW, WINDOW,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    ax_rgb.add_patch(rect)
    ax_rgb.text(x + 6, y + 18, str(i), color="yellow", fontsize=10, weight="bold")

plt.tight_layout()
plt.show()
fig_rgb.savefig("rgb_sliding_windows.png", dpi=300, bbox_inches="tight")
plt.close(fig_rgb)


# Mask image with crop windows
fig_mask, ax_mask = plt.subplots(figsize=(10, 10))
ax_mask.imshow(mask_vis, cmap="gray")
ax_mask.axis("off")

for (y, x) in origins:
    rect = patches.Rectangle(
        (x, y), WINDOW, WINDOW,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    ax_mask.add_patch(rect)

plt.tight_layout()
plt.show()
fig_mask.savefig("mask_sliding_windows.png", dpi=300, bbox_inches="tight")
plt.close(fig_mask)

In [ ]:
# Combined RGB and mask view with the same sliding windows
mask_vis = (train_mask_full > 0).astype(np.uint8)

fig, axes = plt.subplots(2, 1, figsize=(12, 14), sharex=True, sharey=True)

axes[0].imshow(train_image_full)
axes[0].set_title(f"RGB — Sliding windows {WINDOW}x{WINDOW} (stride={STRIDE})")
axes[0].axis("off")

axes[1].imshow(mask_vis, cmap="gray")
axes[1].set_title("MASK — Sliding windows")
axes[1].axis("off")

for i, (y, x) in enumerate(origins):
    for ax in axes:
        rect = patches.Rectangle(
            (x, y), WINDOW, WINDOW,
            linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)

    axes[0].text(x + 6, y + 18, str(i), color="yellow", fontsize=10, weight="bold")

plt.tight_layout()
plt.show()

In [ ]:
def count_crops_for_pair(img_path: Path, mask_path: Path, win=512, stride=512, tamper_thr=0.0):
    """
    Count tampered and clean crops for a single (image, mask) pair.
    A crop is considered tampered if the tampered-pixel ratio is above tamper_thr.
    """
    
    with Image.open(img_path) as im:
        W, H = im.size

    m = np.array(Image.open(mask_path), dtype=np.int16)
    m = (m > 0).astype(np.int16)

    if m.shape[0] != H or m.shape[1] != W:
        raise RuntimeError(f"Mask/image size mismatch: {img_path.name} img=({H},{W}) mask={m.shape}")

    tampered = 0
    clean = 0
    total = 0

    for y, x in sliding_origins(H, W, win=win, stride=stride):
        mc = m[y:y+win, x:x+win]
        denom = mc.size
        if denom == 0:
            continue

        ratio = float((mc == 1).sum()) / float(denom)
        total += 1

        if ratio > tamper_thr:
            tampered += 1
        else:
            clean += 1

    return {"total": total, "tampered": tampered, "clean": clean, "H": H, "W": W}

In [ ]:
def count_crops_dataset(pairs, win=512, stride=512, tamper_thr=0.0, types=TYPES, desc="Counting crops"):
    """
    Count crops globally and per document type across the whole dataset.
    """
    
    totals = Counter()
    by_type_total = Counter()
    by_type_tampered = Counter()
    by_type_clean = Counter()
    per_doc = []

    for img_path, mask_path in tqdm(pairs, desc=desc, total=len(pairs)):
        doc_type = infer_type_from_stem(img_path.stem, types)

        c = count_crops_for_pair(
            img_path=img_path,
            mask_path=mask_path,
            win=win,
            stride=stride,
            tamper_thr=tamper_thr,
        )

        totals["docs"] += 1
        totals["crops_total"] += c["total"]
        totals["crops_tampered"] += c["tampered"]
        totals["crops_clean"] += c["clean"]

        by_type_total[doc_type] += c["total"]
        by_type_tampered[doc_type] += c["tampered"]
        by_type_clean[doc_type] += c["clean"]

        per_doc.append({
            "stem": img_path.stem,
            "type": doc_type,
            "H": c["H"],
            "W": c["W"],
            "crops_total": c["total"],
            "crops_tampered": c["tampered"],
            "crops_clean": c["clean"],
        })

    return {
        "totals": totals,
        "by_type_total": by_type_total,
        "by_type_tampered": by_type_tampered,
        "by_type_clean": by_type_clean,
        "per_doc": per_doc,
    }

In [ ]:
# Count crop statistics across the full dataset
stats_all = count_crops_dataset(
    pairs,
    win=WINDOW,
    stride=STRIDE,
    tamper_thr=TAMPER_THR,
    types=TYPES,
    desc=f"All pairs (win={WINDOW}, stride={STRIDE}, thr={TAMPER_THR})"
)

print("\n=== GLOBAL (ALL) ===")
print(f"Docs:           {stats_all['totals']['docs']}")
print(f"Crops total:    {stats_all['totals']['crops_total']}")
print(f"Crops tampered: {stats_all['totals']['crops_tampered']}")
print(f"Crops clean:    {stats_all['totals']['crops_clean']}")

if stats_all['totals']['crops_total'] > 0:
    print(f"Tampered ratio: {stats_all['totals']['crops_tampered'] / stats_all['totals']['crops_total']:.4f}")

print("\n=== CROPS PER TYPE ===")
for t in TYPES + ["other"]:
    if t in stats_all["by_type_total"]:
        print(
            f"{t:10s}  total={stats_all['by_type_total'][t]:8d}  "
            f"tampered={stats_all['by_type_tampered'][t]:8d}  "
            f"clean={stats_all['by_type_clean'][t]:8d}"
        )

In [ ]:
def estimate_pixel_pos_weight_from_pairs(pairs, win=512, stride=512, max_docs=None):
    """
    Estimate pixel-level pos_weight = negative_pixels / positive_pixels
    using the sliding-window setup.
    """
    
    pos = 0
    neg = 0

    it = pairs if max_docs is None else pairs[:max_docs]
    for img_path, mask_path in tqdm(it, desc=f"Estimating pos_weight (win={win}, stride={stride})", total=len(it)):
        m = np.array(Image.open(mask_path), dtype=np.int16)
        m = (m > 0).astype(np.uint8)

        H, W = m.shape
        origins = sliding_origins(H, W, win=win, stride=stride)

        for y, x in origins:
            mc = m[y:y+win, x:x+win]
            pos += int(mc.sum())
            neg += int(mc.size - mc.sum())

    pos_weight = neg / max(pos, 1)
    return {"pos_pixels": pos, "neg_pixels": neg, "pos_weight": float(pos_weight)}

In [ ]:
# Estimate class imbalance at pixel level
pw_stats = estimate_pixel_pos_weight_from_pairs(
    pairs,
    win=WINDOW,
    stride=STRIDE,
    max_docs=None,
)

print("\n=== Pixel stats (sliding window) ===")
print(f"pos_pixels: {pw_stats['pos_pixels']}")
print(f"neg_pixels: {pw_stats['neg_pixels']}")
print(f"pos_weight (neg/pos): {pw_stats['pos_weight']:.2f}")

In [ ]:
def infer_type(stem: str, types=TYPES):
    """
    Infer document type from stem using known RTM prefixes.
    """
    
    for t in types:
        if stem.startswith(t):
            return t
    return "other"


def split_stratified_by_type(pairs, val_ratio=0.10, test_ratio=0.10, seed=42):
    """
    Split the dataset by document, stratified by document type prefix.
    No documents are discarded.
    """
    
    rng = random.Random(seed)
    buckets = defaultdict(list)

    for img, mask in pairs:
        buckets[infer_type(img.stem)].append((img, mask))

    train_split, val_split, test_split = [], [], []

    for t, items in buckets.items():
        rng.shuffle(items)
        n = len(items)

        n_val = int(round(val_ratio * n))
        n_test = int(round(test_ratio * n))
        n_train = n - n_val - n_test

        val_split += items[:n_val]
        test_split += items[n_val:n_val+n_test]
        train_split += items[n_val+n_test:]

    rng.shuffle(train_split)
    rng.shuffle(val_split)
    rng.shuffle(test_split)

    def summarize(split, name):
        c = Counter(infer_type(img.stem) for img, _ in split)
        n_good = c.get("good_", 0)
        n_tamp = len(split) - n_good

        print(f"\n{name}: {len(split)} docs  (good={n_good}, tampered={n_tamp})")
        for k in TYPES + ["other"]:
            if k in c:
                print(f"  {k:10s}: {c[k]}")

    summarize(train_split, "TRAIN")
    summarize(val_split, "VAL")
    summarize(test_split, "TEST")

    return train_split, val_split, test_split

In [ ]:
# Create train/validation/test splits
train_split, val_split, test_split = split_stratified_by_type(
    pairs,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=42,
)

In [ ]:
total_pairs = len(pairs)

print("SPLIT SIZES\n")
print(f"Total pairs: {total_pairs}")
print(f"Train: {len(train_split)}  ({len(train_split)/total_pairs*100:.2f}%)")
print(f"Val:   {len(val_split)}    ({len(val_split)/total_pairs*100:.2f}%)")
print(f"Test:  {len(test_split)}   ({len(test_split)/total_pairs*100:.2f}%)")
print("\nTotal across splits:", len(train_split) + len(val_split) + len(test_split))

In [ ]:
def materialize_split(split_files, root_dir: Path, split_name: str, use_symlink=False):
    """
    Create a physical split directory and copy/symlink the corresponding
    images and masks into:
        <root_dir>/<split_name>/JPEGImages
        <root_dir>/<split_name>/SegmentationClass
    """
    
    split_root = root_dir / split_name
    img_out_dir = split_root / "JPEGImages"
    mask_out_dir = split_root / "SegmentationClass"

    img_out_dir.mkdir(parents=True, exist_ok=True)
    mask_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nMaterializing split '{split_name}' in {split_root}")

    for img_path, mask_path in tqdm(split_files, desc=f"{split_name}"):
        dst_img = img_out_dir / img_path.name
        dst_mask = mask_out_dir / mask_path.name

        if not dst_img.exists():
            if use_symlink:
                dst_img.symlink_to(img_path)
            else:
                shutil.copy2(img_path, dst_img)

        if not dst_mask.exists():
            if use_symlink:
                dst_mask.symlink_to(mask_path)
            else:
                shutil.copy2(mask_path, dst_mask)

    print(f"✅ {split_name.upper()} done: {len(split_files)} images + masks")

In [25]:
# Materialize the final dataset splits on disk
materialize_split(train_split, REALTEXT_ROOT, "train_v2", use_symlink=False)
materialize_split(val_split, REALTEXT_ROOT, "val_v2", use_symlink=False)
materialize_split(test_split, REALTEXT_ROOT, "test_v2", use_symlink=False)


📂 A materializar split 'train_v2' em /media/general_storage6/rmastorage/datasets/RealTextManipulation/train_v2


train_v2: 100%|█████████████████████████████████████████████████████████████████████| 7200/7200 [10:37<00:00, 11.30it/s]


✅ TRAIN_V2 concluído: 7200 imagens + máscaras

📂 A materializar split 'val_v2' em /media/general_storage6/rmastorage/datasets/RealTextManipulation/val_v2


val_v2: 100%|█████████████████████████████████████████████████████████████████████████| 900/900 [01:26<00:00, 10.38it/s]


✅ VAL_V2 concluído: 900 imagens + máscaras

📂 A materializar split 'test_v2' em /media/general_storage6/rmastorage/datasets/RealTextManipulation/test_v2


test_v2: 100%|████████████████████████████████████████████████████████████████████████| 900/900 [01:18<00:00, 11.51it/s]

✅ TEST_V2 concluído: 900 imagens + máscaras


In [26]:
# Sanity check: count files in each materialized split
for split in ["train_v2", "val_v2", "test_v2"]:
    img_dir = REALTEXT_ROOT / split / "JPEGImages"
    mask_dir = REALTEXT_ROOT / split / "SegmentationClass"

    print(f"\n{split.upper()}:")
    print("  Images:", len(list(img_dir.glob("*"))))
    print("  Masks:", len(list(mask_dir.glob("*"))))


TRAIN_V2:
  Imagens: 7200
  Máscaras: 7200

VAL_V2:
  Imagens: 900
  Máscaras: 900

TEST_V2:
  Imagens: 900
  Máscaras: 900


In [ ]:
def show_random_image_size(split_name):
    """
    Show one random image filename and resolution from a split.
    """
    
    img_dir = REALTEXT_ROOT / split_name / "JPEGImages"
    images = list(img_dir.glob("*"))

    if len(images) == 0:
        print(f"No images found in {split_name}")
        return

    img_path = random.choice(images)

    with Image.open(img_path) as img:
        width, height = img.size

    print(f"{split_name.upper()}")
    print(f"  File: {img_path.name}")
    print(f"  Size: {width} x {height}\n")

In [28]:
show_random_image_size("train_v2")
show_random_image_size("val_v2")
show_random_image_size("test_v2")

TRAIN_V2
  Ficheiro: insert_0135.jpg
  Tamanho: 1204 x 1718

VAL_V2
  Ficheiro: cover_0579.jpg
  Tamanho: 1280 x 1706

TEST_V2
  Ficheiro: cover_0689.jpg
  Tamanho: 1752 x 1240



In [ ]:
def max_hw_in_split(split_name):
    """
    Find the maximum width and height observed in a split.
    """
    
    img_dir = REALTEXT_ROOT / split_name / "JPEGImages"

    max_w = 0
    max_h = 0
    max_img = None

    images = list(img_dir.glob("*"))

    for img_path in tqdm(images, desc=f"Analyzing {split_name}"):
        try:
            with Image.open(img_path) as img:
                w, h = img.size
        except Exception as e:
            print(f"⚠️ Error opening {img_path.name}: {e}")
            continue

        if w > max_w or h > max_h:
            max_w = max(max_w, w)
            max_h = max(max_h, h)
            max_img = img_path.name

    print(f"\n{split_name.upper()}")
    print(f"  Max width : {max_w}")
    print(f"  Max height: {max_h}")
    print(f"  Example image: {max_img}\n")

    return max_w, max_h

In [31]:
max_hw_in_split("train_v2")
max_hw_in_split("val_v2")
max_hw_in_split("test_v2")

Analisando train_v2: 100%|█████████████████████████████████████████████████████████| 7200/7200 [00:27<00:00, 260.00it/s]



TRAIN_V2
  Máx Width : 2000
  Máx Height: 2000
  Exemplo imagem: good_1652.jpg



Analisando val_v2: 100%|█████████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 282.56it/s]



VAL_V2
  Máx Width : 2000
  Máx Height: 2000
  Exemplo imagem: good_1270.jpg



Analisando test_v2: 100%|████████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 273.23it/s]


TEST_V2
  Máx Width : 2000
  Máx Height: 2000
  Exemplo imagem: good_0787.jpg



(2000, 2000)

In [ ]:
# Height histogram bins
H_BINS = [
    (0, 512),
    (512, 1024),
    (1024, 1536),
    (1536, 2000),
]

In [ ]:
def height_distribution(split_name, bins):
    """
    Print the height distribution of images in a split.
    """
    
    img_dir = REALTEXT_ROOT / split_name / "JPEGImages"
    counter = Counter()

    for img_path in tqdm(list(img_dir.glob("*")), desc=f"{split_name}"):
        try:
            with Image.open(img_path) as img:
                _, h = img.size
        except Exception as e:
            print(f"Error in {img_path.name}: {e}")
            continue

        matched = False
        for low, high in bins:
            if low <= h < high:
                counter[f"{low}-{high}"] += 1
                matched = True
                break

        if not matched:
            counter[f">={bins[-1][1]}"] += 1

    print(f"\nHEIGHT DISTRIBUTION — {split_name.upper()}")
    total = sum(counter.values())
    for k in counter:
        pct = counter[k] / total * 100
        print(f"  {k:>10}: {counter[k]:5d} ({pct:5.2f}%)")

    return counter

In [35]:
height_distribution("train_v2", H_BINS)
height_distribution("val_v2", H_BINS)
height_distribution("test_v2", H_BINS)

train_v2: 100%|████████████████████████████████████████████████████████████████████| 7200/7200 [00:11<00:00, 633.07it/s]



📊 DISTRIBUIÇÃO DE HEIGHT — TRAIN_V2
   1536-2000:  2075 (28.82%)
    512-1024:  2570 (35.69%)
   1024-1536:  2399 (33.32%)
      >=2000:   156 ( 2.17%)


val_v2: 100%|████████████████████████████████████████████████████████████████████████| 900/900 [00:01<00:00, 511.76it/s]



📊 DISTRIBUIÇÃO DE HEIGHT — VAL_V2
    512-1024:   329 (36.56%)
   1024-1536:   295 (32.78%)
   1536-2000:   251 (27.89%)
      >=2000:    25 ( 2.78%)


test_v2: 100%|███████████████████████████████████████████████████████████████████████| 900/900 [00:01<00:00, 484.24it/s]


📊 DISTRIBUIÇÃO DE HEIGHT — TEST_V2
   1024-1536:   300 (33.33%)
   1536-2000:   262 (29.11%)
    512-1024:   320 (35.56%)
      >=2000:    18 ( 2.00%)


Counter({'512-1024': 320, '1024-1536': 300, '1536-2000': 262, '>=2000': 18})

In [36]:
# Width histogram bins
W_BINS = [
    (0, 512),
    (512, 1024),
    (1024, 1536),
    (1536, 2000),
]

In [ ]:
def width_distribution(split_name, bins):
    """
    Print the width distribution of images in a split.
    """
    
    img_dir = REALTEXT_ROOT / split_name / "JPEGImages"
    counter = Counter()

    for img_path in tqdm(list(img_dir.glob("*")), desc=f"{split_name}"):
        try:
            with Image.open(img_path) as img:
                w, _ = img.size
        except Exception as e:
            print(f"Error in {img_path.name}: {e}")
            continue

        matched = False
        for low, high in bins:
            if low <= w < high:
                counter[f"{low}-{high}"] += 1
                matched = True
                break

        if not matched:
            counter[f">={bins[-1][1]}"] += 1

    print(f"\nWIDTH DISTRIBUTION — {split_name.upper()}")
    total = sum(counter.values())
    for k in counter:
        pct = counter[k] / total * 100
        print(f"  {k:>10}: {counter[k]:5d} ({pct:5.2f}%)")

    return counter

In [38]:
width_distribution("train_v2", W_BINS)
width_distribution("val_v2", W_BINS)
width_distribution("test_v2", W_BINS)

train_v2: 100%|████████████████████████████████████████████████████████████████████| 7200/7200 [00:07<00:00, 943.89it/s]



📊 DISTRIBUIÇÃO DE HEIGHT — TRAIN_V2
   1024-1536:  2343 (32.54%)
    512-1024:  4105 (57.01%)
   1536-2000:   698 ( 9.69%)
      >=2000:    54 ( 0.75%)


val_v2: 100%|████████████████████████████████████████████████████████████████████████| 900/900 [00:01<00:00, 472.24it/s]



📊 DISTRIBUIÇÃO DE HEIGHT — VAL_V2
    512-1024:   494 (54.89%)
   1024-1536:   303 (33.67%)
   1536-2000:    98 (10.89%)
      >=2000:     5 ( 0.56%)


test_v2: 100%|███████████████████████████████████████████████████████████████████████| 900/900 [00:02<00:00, 441.19it/s]


📊 DISTRIBUIÇÃO DE HEIGHT — TEST_V2
   1024-1536:   301 (33.44%)
    512-1024:   485 (53.89%)
      >=2000:    11 ( 1.22%)
   1536-2000:   103 (11.44%)


Counter({'512-1024': 485, '1024-1536': 301, '1536-2000': 103, '>=2000': 11})

In [ ]:
# Example mask inspection from the test split
p = REALTEXT_ROOT / "test_v2/SegmentationClass/edit_0188.png"
m = np.array(Image.open(p))

print("dtype:", m.dtype)
print("shape:", m.shape)
u = np.unique(m)
print("unique values (first 50):", u[:50], " ... total:", len(u))
print("min/max:", m.min(), m.max())

In [ ]:
def create_split_txt(root_dir: Path, split_name: str, output_dir: Path):
    """
    Create a <split_name>.txt file listing the common stems present in both
    JPEGImages and SegmentationClass for a given split.
    """
    
    jpeg_dir = root_dir / split_name / "JPEGImages"
    mask_dir = root_dir / split_name / "SegmentationClass"

    assert jpeg_dir.exists(), f"Missing directory: {jpeg_dir}"
    assert mask_dir.exists(), f"Missing directory: {mask_dir}"

    images = {p.stem for p in jpeg_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg"]}
    masks = {p.stem for p in mask_dir.iterdir() if p.suffix.lower() == ".png"}

    common = sorted(images & masks)

    print(f"[{split_name}] images={len(images)} masks={len(masks)} common={len(common)}")

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{split_name}.txt"

    with open(out_path, "w") as f:
        for name in common:
            f.write(name + "\n")

    print(f"File created: {out_path}")

In [ ]:
# Generate split index files used by training/evaluation pipelines
OUTPUT = REALTEXT_ROOT

for split in ["train_v2", "val_v2", "test_v2"]:
    create_split_txt(REALTEXT_ROOT, split, OUTPUT)